# One Look · 03 · Baselines et réseau léger (Colab, GPU)

Ce notebook mesure les approches de référence, entraîne le réseau léger, l'évalue sur le test et range les résultats dans un dépôt **privé** Hugging Face, d'où on les récupère en local.

**Avant de lancer :**

1. *Exécution → Modifier le type d'exécution → GPU (T4)*.
2. Secret Colab `HF_TOKEN` : une clé Hugging Face en **écriture** (pour ranger les résultats), accès activé pour ce notebook.

Durée indicative sur T4 : environ 5 min de préparation, 5 à 10 min de baselines, 15 à 30 min d'entraînement.

In [ ]:
HF_DATA = "Elfried9684/onelook-alert-data"
HF_RUNS = "Elfried9684/onelook-alert-runs"
REPO_URL = "https://github.com/elfried96/onelook-alert_Elf.git"
WORKDIR = "/content/onelook-alert_Elf"
ARCH, SIZE, EPOCHS = "mobilenet_v3_small", 256, 15

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : changer le type d'exécution"

## 1. Code, environnement et données

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull -q
%cd {WORKDIR}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --frozen --group train
!git log --oneline -1
!uv run hf download {HF_DATA} --repo-type dataset --local-dir data/pack --quiet
!uv run python scripts/check_pack.py

## 2. Baselines

Seuils NDWI / NBR choisis sur l'entraînement, puis classifieur sur statistiques par bande. Mesures sur la validation et le test.

In [ ]:
!uv run python scripts/baselines.py

## 3. Réseau léger

Échantillonnage équilibré (chaque décision pèse autant ; parmi les normales, la Californie pèse autant que le reste du monde), rotations et miroirs. La meilleure époque est choisie sur le F1 macro de validation, puis le seuil « inconnu » est calibré sur la validation.

In [ ]:
RUN = f"{ARCH}_{SIZE}"
!uv run python scripts/train.py --arch {ARCH} --size {SIZE} --epochs {EPOCHS}

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
history = pd.DataFrame(json.load(open(f"runs/{RUN}/history.json")))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history.epoque, history.perte); ax[0].set_title("Perte d'entraînement"); ax[0].set_xlabel("époque")
for col, name in [("f1_macro", "F1 macro"), ("rappel_inondation", "rappel inondation"),
                  ("rappel_incendie", "rappel incendie"), ("fausses_alertes", "fausses alertes")]:
    ax[1].plot(history.epoque, history[col], label=name)
ax[1].set_title("Validation"); ax[1].set_xlabel("époque"); ax[1].legend(frameon=False); plt.show()

## 4. Évaluation sur le test (ONNX, CPU)

In [ ]:
!uv run python scripts/evaluate.py runs/{RUN} --split test

## 5. Ranger les résultats

Dépôt **privé** sur Hugging Face : modèle ONNX, configuration, courbes, décisions et poids. Rien de cela ne va dans Git.

In [ ]:
!uv run hf repos create {HF_RUNS} --type model --private --exist-ok
!uv run hf upload {HF_RUNS} runs runs --quiet
print(f"https://huggingface.co/{HF_RUNS}/tree/main/runs")

Ensuite, en local : `uv run hf download Elfried9684/onelook-alert-runs --include "runs/*" --local-dir .` récupère `runs/`, pour l'évaluation, la page Résultats et la démo.